# Transaction types

For each type of securities lending transaction, what its row should show, followed by three examples from the cleaned table on the latest reference period. The collateral column lists the ISINs of securities collateral or the currency and amount of cash collateral.

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

Which types are there and how frequent are they?

In [ ]:
query = f"""

SELECT collateral_type, COUNT(*) AS n
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
GROUP BY 1
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

## Cash collateral

The borrower posts cash, the lender pays a rebate rate on it and reinvests the cash. Expect a rebate rate, no lending fee, and currency plus amount in the collateral column. A lending fee instead of a rebate marks a cash pool loan, where the cash is fee based.

In [ ]:
query = f"""

SELECT x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
       x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
       x.collateral_type,
       COALESCE(GROUP_CONCAT(CASE WHEN c.collateral_kind = 'cash'
                                  THEN CONCAT(c.cash_currency, ' ', CAST(ROUND(c.cash_amount) AS STRING))
                                  ELSE c.collateral_isin END, ', '),
                x.collateral_basket_id) AS collateral,
       x.lending_fee, x.rebate_rate
FROM (
  SELECT *
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
    AND collateral_type = 'cash'
  ORDER BY uti
  LIMIT 3
) x
LEFT JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
  ON c.tec_ruti = x.tec_ruti AND c.reference_period = x.reference_period
GROUP BY x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
         x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
         x.collateral_type, x.collateral_basket_id, x.lending_fee, x.rebate_rate
ORDER BY x.lender_id

"""
df = pd.read_sql_query(query, cnxn)
df

## Securities collateral

The borrower posts other securities and pays a lending fee. Expect a lending fee, no rebate rate, and a list of ISINs in the collateral column.

In [ ]:
query = f"""

SELECT x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
       x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
       x.collateral_type,
       COALESCE(GROUP_CONCAT(CASE WHEN c.collateral_kind = 'cash'
                                  THEN CONCAT(c.cash_currency, ' ', CAST(ROUND(c.cash_amount) AS STRING))
                                  ELSE c.collateral_isin END, ', '),
                x.collateral_basket_id) AS collateral,
       x.lending_fee, x.rebate_rate
FROM (
  SELECT *
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
    AND collateral_type = 'securities'
  ORDER BY uti
  LIMIT 3
) x
LEFT JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
  ON c.tec_ruti = x.tec_ruti AND c.reference_period = x.reference_period
GROUP BY x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
         x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
         x.collateral_type, x.collateral_basket_id, x.lending_fee, x.rebate_rate
ORDER BY x.lender_id

"""
df = pd.read_sql_query(query, cnxn)
df

## Mixed collateral

Cash and securities posted against the same loan. Rare. Expect both kinds in the collateral column and either a fee or a rebate.

In [ ]:
query = f"""

SELECT x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
       x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
       x.collateral_type,
       COALESCE(GROUP_CONCAT(CASE WHEN c.collateral_kind = 'cash'
                                  THEN CONCAT(c.cash_currency, ' ', CAST(ROUND(c.cash_amount) AS STRING))
                                  ELSE c.collateral_isin END, ', '),
                x.collateral_basket_id) AS collateral,
       x.lending_fee, x.rebate_rate
FROM (
  SELECT *
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
    AND collateral_type = 'mixed'
  ORDER BY uti
  LIMIT 3
) x
LEFT JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
  ON c.tec_ruti = x.tec_ruti AND c.reference_period = x.reference_period
GROUP BY x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
         x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
         x.collateral_type, x.collateral_basket_id, x.lending_fee, x.rebate_rate
ORDER BY x.lender_id

"""
df = pd.read_sql_query(query, cnxn)
df

## Basket collateral

The collateral is defined by an agreed basket and the pieces are not listed on the loan. Expect a lending fee and the ISIN of the basket in the collateral column.

In [ ]:
query = f"""

SELECT x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
       x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
       x.collateral_type,
       COALESCE(GROUP_CONCAT(CASE WHEN c.collateral_kind = 'cash'
                                  THEN CONCAT(c.cash_currency, ' ', CAST(ROUND(c.cash_amount) AS STRING))
                                  ELSE c.collateral_isin END, ', '),
                x.collateral_basket_id) AS collateral,
       x.lending_fee, x.rebate_rate
FROM (
  SELECT *
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
    AND collateral_type = 'basket'
  ORDER BY uti
  LIMIT 3
) x
LEFT JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
  ON c.tec_ruti = x.tec_ruti AND c.reference_period = x.reference_period
GROUP BY x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
         x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
         x.collateral_type, x.collateral_basket_id, x.lending_fee, x.rebate_rate
ORDER BY x.lender_id

"""
df = pd.read_sql_query(query, cnxn)
df

## Net exposure

The collateral covers all loans between the two counterparties as a pool and is not allocated to the loan, so the collateral column is empty. Typical for agency lending, so an agent lender is usually present. Expect a lending fee, a rebate rate instead points to a cash pool.

In [ ]:
query = f"""

SELECT x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
       x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
       x.collateral_type,
       COALESCE(GROUP_CONCAT(CASE WHEN c.collateral_kind = 'cash'
                                  THEN CONCAT(c.cash_currency, ' ', CAST(ROUND(c.cash_amount) AS STRING))
                                  ELSE c.collateral_isin END, ', '),
                x.collateral_basket_id) AS collateral,
       x.lending_fee, x.rebate_rate
FROM (
  SELECT *
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
    AND collateral_type = 'net_exposure'
  ORDER BY uti
  LIMIT 3
) x
LEFT JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
  ON c.tec_ruti = x.tec_ruti AND c.reference_period = x.reference_period
GROUP BY x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
         x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
         x.collateral_type, x.collateral_basket_id, x.lending_fee, x.rebate_rate
ORDER BY x.lender_id

"""
df = pd.read_sql_query(query, cnxn)
df

## Uncollateralised

No collateral at all. Expect a lending fee, no rebate rate and an empty collateral column.

In [ ]:
query = f"""

SELECT x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
       x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
       x.collateral_type,
       COALESCE(GROUP_CONCAT(CASE WHEN c.collateral_kind = 'cash'
                                  THEN CONCAT(c.cash_currency, ' ', CAST(ROUND(c.cash_amount) AS STRING))
                                  ELSE c.collateral_isin END, ', '),
                x.collateral_basket_id) AS collateral,
       x.lending_fee, x.rebate_rate
FROM (
  SELECT *
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
    AND collateral_type = 'none'
  ORDER BY uti
  LIMIT 3
) x
LEFT JOIN xlab_ecb_prj_sftds_cb_common.hermesf_sl_collateral c
  ON c.tec_ruti = x.tec_ruti AND c.reference_period = x.reference_period
GROUP BY x.reference_period, x.lender_id, x.borrower_id, x.agent_lender_id,
         x.isin, x.loan_security_type, x.loan_quantity, x.loan_value_eur, x.loan_value_currency,
         x.collateral_type, x.collateral_basket_id, x.lending_fee, x.rebate_rate
ORDER BY x.lender_id

"""
df = pd.read_sql_query(query, cnxn)
df

## Diagnostics

How often is an agent lender involved, per transaction type?

In [ ]:
query = f"""

SELECT collateral_type, COUNT(*) AS n,
       COUNT(agent_lender_id) AS n_agent,
       ROUND(100 * COUNT(agent_lender_id) / COUNT(*), 1) AS pct_agent
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
WHERE reference_period = (SELECT MAX(reference_period) FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl)
GROUP BY 1
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

Loan volume over time, split by whether an agent lender is involved. Volume is the sum of `loan_value_eur` over the loans outstanding on each day.

In [ ]:
query = f"""

SELECT reference_period,
       CASE WHEN agent_lender_id IS NOT NULL THEN 'agent lender' ELSE 'no agent lender' END AS agent,
       SUM(loan_value_eur) / 1000000000 AS volume_bn_eur
FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
GROUP BY 1, 2
ORDER BY 1, 2

"""
df = pd.read_sql_query(query, cnxn)
df

In [ ]:
df['reference_period'] = pd.to_datetime(df['reference_period'])
ts = df.pivot(index='reference_period', columns='agent', values='volume_bn_eur')
ax = ts.plot(figsize=(10, 4), linewidth=2, color=['#2a78d6', '#eb6834'])
ax.set_title('Loan volume by agent lender involvement')
ax.set_ylabel('EUR bn')
ax.set_xlabel('')
ax.grid(axis='y', color='#e5e5e5')
ax.legend(frameon=False)